# Pokémon EDA: Group Activities

Ten group activities that apply pandas to **exploratory data analysis (EDA)** on `Pokemon.csv`. They build on [pandas_fundamentals.ipynb](pandas_fundamentals.ipynb); each activity lists the sections it uses, and [pandas_cheatsheet.md](pandas_cheatsheet.md) has every method you'll need.

### The dataset
800 rows, one per Pokémon **or alternate form**, from generations 1–6.

| Column | Meaning |
|---|---|
| `#` | Pokédex number (⚠️ not unique) |
| `Name` | name; alternate forms are glued onto the base name |
| `Type 1`, `Type 2` | primary and secondary type (`Type 2` is empty for single-type Pokémon) |
| `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` | the six base stats |
| `Total` | sum of the six stats |
| `Generation` | game generation, 1–6 |
| `Legendary` | `True` / `False` |


**Rules for every answer**
1. Back each claim with **one table or number** from pandas.
2. State **one caveat**: something in the data that could make the claim misleading.
3. There is rarely one right answer. Be ready to explain *why* you chose your approach, filters and thresholds.

Activities 1 and 2 clean the data that later activities rely on. If you skip them, run the **checkpoint cell** after Activity 2.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)

df = pd.read_csv("Pokemon.csv")
STATS = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

df.head()

---
## Activity 1: Data detective 🔍
**Uses:** §3 inspecting, §7 missing data, §9 counting

Before analysing anything, find out what you're dealing with. Your group must report **three things that are wrong with, or surprising about, this dataset.**

Use only: `shape`, `info`, `describe`, `isna`, `nunique`, `duplicated`, `value_counts`, `head`/`sample`.

Then answer:
1. `#` has fewer distinct values than there are rows. Why? Look at some of the duplicated rows.
2. `Type 2` has many missing values. Is that bad data, or does "missing" mean something here?
3. Can you **prove** that `Total` is the sum of the six stats for every row?
4. How many legendaries are there, and how different are their stats from everyone else's?

In [ ]:
# your code here

---
## Activity 2: Clean the names ✂️
**Uses:** §8 `.str` methods & regex, §5 new columns, §11 merge

Alternate forms have names like `VenusaurMega Venusaur`, `CharizardMega Charizard X` or `KyogrePrimal Kyogre`: the base name and the form name are glued together.

1. Split `Name` into `base_name` (`Venusaur`) and `form` (`Mega Venusaur`, or missing for normal Pokémon).
   *Hint:* the glue point is a **lowercase letter immediately followed by an uppercase letter**. `s.str.extract(regex)` returns one column per capture group, and `(?P<name>...)` names a group.
2. Add a boolean `is_mega` column. 🔮 *Predict:* would `df["Name"].str.contains("Mega")` give the right answer? Check it against your column.
3. Check your work: list every distinct `form` that is **not** a Mega. Does your regex break any normal names (try `Mr. Mime`, `Ho-oh`, `Porygon-Z`)?
4. **Bonus:** for each Mega, how many `Total` points does it gain over its base form? Merge the Megas back onto their base forms (a *self-merge*).

In [ ]:
# your code here

### ✅ Checkpoint
Run this if you skipped Activity 2, or if your version of the columns differs. Later activities rely on `base_name`, `form` and `is_mega`.

> ⚠️ **This cell contains one answer to Activity 2.** Finish (or give up on) Activity 2 before you read it.

In [ ]:
parts = df["Name"].str.extract(r"^(?P<base_name>.*?[a-z])(?P<form>[A-Z].*)$")
df["base_name"] = parts["base_name"].fillna(df["Name"])
df["form"] = parts["form"]
df["is_mega"] = df["form"].str.contains("Mega", na=False)
df[["Name", "base_name", "form", "is_mega"]].sample(5, random_state=1)

---
## Activity 3: Myth or fact? ⚖️
**Uses:** §4 filtering, §10 groupby

**Scoring:** 1 point for a correct verdict backed by evidence · +1 for a caveat · +1 for checking whether the result **still holds without legendaries and Megas**.

| Card | Claim |
|---|---|
| A | *Legendary Pokémon are faster than non-legendaries.* |
| B | *Dragon is the strongest type.* |
| C | *Bug is the weakest type.* |
| D | *Dual-type Pokémon have a higher Total than single-type ones.* |
| E | *Pokémon with high Attack tend to have low Defense (a trade-off).* |
| F | *Water is the most common type.* |

In [ ]:
# Card: ___
# your code here

---
## Activity 4: Power creep across generations 📈
**Uses:** §10 groupby, §12 pivot_table

*"Newer generations are stronger."* Is that true, or is it an artefact of which Pokémon we include?

1. Build **one table** with the mean `Total` per `Generation`, computed three ways (three columns):
   - all rows
   - without legendaries
   - without legendaries **and** without Megas
2. Does the story change between the columns? Which version would you present, and why?
3. Make a `pivot_table` of mean `Total` with `Type 1` as rows and `Generation` as columns. Which types show the most change between generations?
4. **Discussion:** some type × generation cells are based on only 1–3 Pokémon. Show the **count** for each cell as well. How should that affect which cells you trust?

In [ ]:
# your code here

---
## Activity 5: Fair type rankings 🔄
**Uses:** §12 melt, §7 dropna, §9 rank

`df.groupby("Type 1")` silently ignores every Pokémon's **second** type. A Water/Flying Pokémon only counts as Water.

1. Reshape the data so each Pokémon appears **once per type it has**: `melt` the `Type 1` and `Type 2` columns into a single `type` column, then drop the rows where the type is missing.
   🔮 *Predict* how many rows the long table will have before you run it.
2. Rank the types by mean `Attack`, `Defense` and `Speed` using the long table (the *fair* ranking).
3. Compare it with the naive `Type 1` ranking. Which types move the most? Why those?

In [ ]:
# your code here

---
## Activity 6: Role classifier 🛡️
**Uses:** §5 new columns & `np.select`, §12 crosstab

Give every Pokémon a **role** based on its stats, for example:

| Role | Idea |
|---|---|
| physical attacker | `Attack` is its best stat |
| special attacker | `Sp. Atk` is its best stat |
| tank | `HP`, `Defense` or `Sp. Def` is its best stat |
| speedster | `Speed` is its best stat |
| balanced | no stat stands out much |

1. Agree on your group's rules. **Write the definition down first**, especially what "balanced" means.
   *Useful tools:* `df[STATS].idxmax(axis=1)` gives the column name of each row's best stat; `df[STATS].max(axis=1) - df[STATS].min(axis=1)` measures the spread.
2. Add a `role` column and count how many Pokémon have each role.
3. `pd.crosstab` role against `Type 1` (try `normalize="index"`). Which types are most "tanky"? Which are speedsters?
4. **Compare across groups.** Different definitions give different answers. Whose is the most defensible?

In [ ]:
# your code here

---
## Activity 7: Join a lookup table 🗺️
**Uses:** §11 merge, `validate`, `indicator`

Each generation's games are set in a region:

| Generation | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Region | Kanto | Johto | Hoenn | Sinnoh | Unova | Kalos |

1. Build this as a small DataFrame and `merge` it onto `df` with `validate="many_to_one"`. Confirm the row count didn't change.
2. Average `Speed` per region, sorted.
3. **Break it on purpose:** add a duplicate row for generation 1 (say, `"Kanto (remake)"`) and merge again with and without `validate`. What happens to the row count and to the averages?
4. Remove generation 6 from the lookup and merge with `how="left", indicator=True`. How would you find the rows that didn't match?

In [ ]:
# your code here

---
## Activity 8: Outlier hunt 🎯
**Uses:** §10 `transform` vs `agg`, §4 masks

1. For each of the six stats, find the Pokémon with the most extreme value. Use a **z-score**: `(value - mean) / std`, the number of standard deviations from the mean.
2. Flag outliers with the **IQR rule**: a value is an outlier if it's below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`, where `IQR = Q3 - Q1`. Which Pokémon are outliers on the **most** stats?
3. Now compute z-scores **within each `Type 1`**: unusual *for its type*. Hint: `groupby("Type 1")[STATS].transform("mean")` gives every row its own type's mean.
4. Find a Pokémon that is unremarkable overall but extreme for its type. Why does `transform` work here while `agg` doesn't?

In [ ]:
# your code here

---
## Activity 9: Relay chain 🏃
**Uses:** §15 method chaining, §4 query, §10 groupby

**One laptop per team.** Build **a single method chain** that answers:

> *Among regular Pokémon (not legendary, not Mega), which **3 primary types** have the highest **median Speed**, counting only types with **at least 20** such Pokémon?*

**Rules**
* Each person adds **exactly one line** to the chain, then passes the laptop on.
* No intermediate variables: one expression from `df` to the answer.
* Anyone may comment out lines from the bottom to debug, but that uses up their turn.
* First team with a correct answer and a readable chain wins.

In [ ]:
(df
    # line 1...
)

---
## Activity 10: Build a squad (capstone) 🏆
**Uses:** everything

Your group drafts a team of **6 Pokémon**. The rules:

1. No legendaries and no Megas.
2. No two members share a `Type 1`.
3. The team's combined `Total` is **at most 3,000**.
4. At least one member has `Speed` ≥ 100.
5. At least one member has `HP` ≥ 100.

Pick a **strategy** (strongest overall? best type coverage? fastest team? best defence?) and **use pandas to choose your picks**, not memory or favourites. Then present:
* the squad (one table),
* your strategy and the pandas steps that produced it,
* one weakness of your squad, backed by data.

Run `check_squad` on your answer before presenting.

In [ ]:
def check_squad(names):
    """Check a list of 6 Pokémon names against the capstone rules."""
    squad = df[df["Name"].isin(names)]
    checks = {
        "6 members, all names found":   len(squad) == 6 and len(names) == 6,
        "no legendaries or Megas":      not (squad["Legendary"] | squad["is_mega"]).any(),
        "unique Type 1":                squad["Type 1"].is_unique,
        "combined Total <= 3000":       squad["Total"].sum() <= 3000,
        "someone with Speed >= 100":    (squad["Speed"] >= 100).any(),
        "someone with HP >= 100":       (squad["HP"] >= 100).any(),
    }
    for rule, ok in checks.items():
        print(("✅ " if ok else "❌ ") + rule)
    print(f"\nCombined Total: {squad['Total'].sum()}")
    return squad[["Name", "Type 1", "Type 2", "Total"] + STATS]

In [ ]:
my_squad = []   # six names, spelled exactly as in the Name column
# check_squad(my_squad)

---
## Wrap-up: gallery walk 🖼️

Each group posts its **three best findings** from the session, each with:
1. the claim in one sentence,
2. the pandas output that supports it,
3. one caveat.

Groups then rotate and leave one question on each other group's findings.

**Debrief questions**
* Which finding changed the most when you removed legendaries or Megas?
* Where did the *shape* of the data (wide vs long) change your answer?
* Which decisions you made (thresholds, definitions, filters) would another analyst disagree with?

*EDA isn't about finding "the answer". It's about knowing which decisions shaped your answer, and saying so.*